In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT))

from eval.encode import encode_words

In [12]:
from pathlib import Path

ROOT = Path.cwd().resolve().parent 
checkpoint_path = str(ROOT / "checkpoints" / "best.pt")

words = ["makan", "memakan", "makanan", "berlari", "permainan"]

kept_words, embeddings = encode_words(
    checkpoint_path=checkpoint_path,
    words=words,
    batch_size=None,
    device="cuda",
)

print("Kept:", kept_words)
print("Embeddings shape:", embeddings.shape)

Kept: ['makan', 'memakan', 'makanan', 'berlari', 'permainan']
Embeddings shape: (5, 768)


In [ ]:
import numpy as np

words = [
    "makan", "memakan", "makanan", "berlari", "permainan",
    "makann", "mkan", "berlarr", "permaenan", "mknn"
]

kept_words, embeddings = encode_words(
    config_path=None,
    checkpoint_path=checkpoint_path,
    words=words,
    batch_size=None,
    device="cuda",
)

# cosine similarity
def cosine_sim(a, b):
    a = a / (np.linalg.norm(a) + 1e-8)
    b = b / (np.linalg.norm(b) + 1e-8)
    return float(np.dot(a, b))

emb = {w: e for w, e in zip(kept_words, embeddings)}

oov_words = ["makann", "mkan", "berlarr", "permaenan", "mknn"]

k = 3
for ow in oov_words:
    if ow not in emb:
        print(f"[SKIP] {ow} tidak ada embedding (mungkin terlalu panjang)")
        continue
    sims = []
    for w, e in emb.items():
        if w == ow:
            continue
        sims.append((w, cosine_sim(emb[ow], e)))
    sims.sort(key=lambda x: x[1], reverse=True)
    topk = sims[:k]
    print(f"\nOOV: {ow}")
    for w, s in topk:
        print(f"  {w:12s}  sim={s:.4f}")



OOV: makann
  makanan       sim=0.9846
  makan         sim=0.9779
  mkan          sim=0.8618

OOV: mkan
  mknn          sim=0.8653
  makan         sim=0.8627
  makann        sim=0.8618

OOV: berlarr
  berlari       sim=0.9857
  permaenan     sim=0.5540
  permainan     sim=0.4860

OOV: permaenan
  permainan     sim=0.9255
  memakan       sim=0.5783
  berlarr       sim=0.5540

OOV: mknn
  mkan          sim=0.8653
  makann        sim=0.6202
  makan         sim=0.5459


In [15]:
from pathlib import Path
ROOT = Path.cwd().resolve().parent  # karena notebook di experiments/
checkpoint_path = str(ROOT / "checkpoints" / "best.pt")

from eval.decode import decode_words

kept, recon = decode_words(
    checkpoint_path=checkpoint_path,
    words=["makan", "memakan", "makanan", "berlari"],
    device="cuda",
)

for w, r in zip(kept, recon):
    print(w, "->", r)


makan -> makan
memakan -> memakan
makanan -> makanan
berlari -> berlari
